## House Price Prediction with MLFLOW

Tasks performed in this project:
* Run hyperparameter tuning while training a model
* Log every hyperparameter and metrics in MLFLOW UI
* Compare the results of the various runs in MLFLOW UI
* Choose the best run and register it as a model

In [9]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()

### 1.Preparing the dataset

In [10]:
data = pd.DataFrame(housing.data,columns= housing.feature_names)
data['Price']=housing.target

### 2. Train test split , Model hyperparameter tuning , MLFLOW Experiments

In [11]:
from urllib.parse import urlparse

# Independent and dependent features
X = data.drop(columns=["Price"])
y = data["Price"]

In [12]:
def hyperparameter_tuning(X_train, y_train, param_grid):
     rf=RandomForestRegressor()
     grid_search= GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2 , scoring="neg_mean_squared_error")
     grid_search.fit(X_train, y_train)

     return grid_search

In [14]:
X_train, X_test, y_train, y_test= train_test_split(X,y, test_size=0.2)


from mlflow.models import infer_signature
signature = infer_signature(X_train, y_train)

# Define hyperparameter grid

param_grid={
    'n_estimators': [100,200],
    'max_depth': [5,10,None],
    'min_samples_split': [2,5],
    'min_samples_leaf': [1,2]
}

mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")
# start mlflow experiments

with mlflow.start_run():
    grid_search=hyperparameter_tuning(X_train, y_train, param_grid)

    best_model= grid_search.best_estimator_

    y_pred= best_model.predict(X_test)
    mse= mean_squared_error(y_test, y_pred)

    mlflow.log_param("best_n_estimators", grid_search.best_params_['n_estimators'])
    mlflow.log_param("best_max_depth", grid_search.best_params_['max_depth'])
    mlflow.log_param("best_min_samples_split", grid_search.best_params_['min_samples_split'])
    mlflow.log_param("best_min_samples_leaf", grid_search.best_params_['min_samples_leaf'])
    mlflow.log_metric("mse", mse)

    tracking_url_type_store=urlparse(mlflow.get_tracking_uri()).scheme


    if tracking_url_type_store != 'file':
        mlflow.sklearn.log_model(best_model,"model", registered_model_name="Best Random Forest Model")
    else:
        mlflow.sklearn.log_model(best_model,"model", signature=signature)

    print(f"Best Hyperparameters: {grid_search.best_params_}")
    print(f"Mean squared error: {mse}")

Fitting 5 folds for each of 24 candidates, totalling 120 fits
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.6s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   2.6s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.7s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   2.8s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.8s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   2.8s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   2.8s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.8s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   2.9s
[CV] END max_depth=5, min_samples_leaf

2026/07/29 19:11:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'Best Random Forest Model'.
2026/07/29 19:11:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Best Random Forest Model, version 1


Best Hyperparameters: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Mean squared error: 0.24363563060416726
🏃 View run learned-snail-335 at: http://127.0.0.1:5000/#/experiments/0/runs/afb3496ddc3f4f94895fdd17b12c3e3b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


Created version '1' of model 'Best Random Forest Model'.
